## NB03 Analysis

In [48]:
import pandas as pd 
import os
from pathlib import Path
import sqlite3
import re
import plotly.graph_objects as go
import plotly.express as px

# Visualise the distribution of comments per post
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from datetime import datetime
from sqlalchemy import create_engine, text

# Import our custom Reddit API module

# --- Configuration for Jupyter ---
# The following magic command is for Jupyter notebooks to render plots inline.
# It should be commented out when running as a standalone script.
%config InlineBackend.figure_formats = ['svg']


In [2]:
engine = create_engine("sqlite:///../data/fantasy_data.db")

**Objective:**
The goal of this section is to determine which drafted players in my league provided the most value in fantasy football by calculating Value Over Replacement Player (VORP). This metric estimates how many more points a player scored compared to a “replacement-level” player at the same position.


**Methodology:**

**Identify Replacement-Level Benchmarks:**

For each position, I calculated the average points per game (PPG) of players near the end of a 14-team league’s starting range. This simulates a “freely available” player.

**Replacement-level ranges:**

QB: Positional ranks 15–18

RB: Ranks 29–32

WR: Ranks 29–32 (including lineupSlotId 4 and 23 for FLEX)

TE: Ranks 15–16

K / DST: Ranks 15–18

**Account for Lineup Slot Variations:**

Some players are listed under different lineupSlotIds depending on their team’s roster setup.

For example, QBs appeared with both slot ID 0 and 20 — both were included in the QB replacement calculation and VORP logic.

**Calculate VORP:**

For each drafted player, I subtracted the replacement-level PPG for their position from their actual PPG.

Players not in a valid draft slot or without enough data received NULL VORP.



In [43]:
## RB = 2, WR = 4, WR = 23, QB = 20, TE = 6, K =17, D/ST = 16
def calculate_all_vorp(engine):
    query = """
    WITH qb_replacement AS (
        SELECT AVG(avg_points) AS ppg FROM players WHERE (lineupSlotId = 20  OR lineupSlotId = 0) AND posRank BETWEEN 15 AND 18
    ),
    rb_replacement AS (
        SELECT AVG(avg_points) AS ppg FROM players WHERE lineupSlotId = 2 AND posRank BETWEEN 29 AND 32
    ),
    wr_replacement AS (
        SELECT AVG(avg_points) AS ppg FROM players WHERE (lineupSlotId = 4  OR lineupSlotId = 23) AND posRank BETWEEN 29 AND 32
    ),
    te_replacement AS (
        SELECT AVG(avg_points) AS ppg FROM players WHERE lineupSlotId = 6 AND posRank BETWEEN 15 AND 16
    ),
    k_replacement AS (
        SELECT AVG(avg_points) AS ppg FROM players WHERE lineupSlotId = 17 AND posRank BETWEEN 15 AND 18
    ),
    dst_replacement AS (
        SELECT AVG(avg_points) AS ppg FROM players WHERE lineupSlotId = 16 AND posRank BETWEEN 15 AND 18
    )

    SELECT 
        d.player_id,
        p.player_name,
        p.points,
        p.position,
        p.avg_points,
        d.overallPickNumber,
        d.lineupSlotId,
        d.roundPickNumber,
        d.roundId,
        ROUND(
            CASE 
                WHEN d.lineupSlotId IN (0,20) THEN p.avg_points - (SELECT ppg FROM qb_replacement)
                WHEN d.lineupSlotId = 2 THEN p.avg_points - (SELECT ppg FROM rb_replacement)
                WHEN d.lineupSlotId IN (4,23) THEN p.avg_points - (SELECT ppg FROM wr_replacement)
                WHEN d.lineupSlotId = 6 THEN p.avg_points - (SELECT ppg FROM te_replacement)
                WHEN d.lineupSlotId = 17 THEN p.avg_points - (SELECT ppg FROM k_replacement)
                WHEN d.lineupSlotId = 16 THEN p.avg_points - (SELECT ppg FROM dst_replacement)
                ELSE NULL
            END
        , 2) AS vorp
    FROM draft d
    JOIN players p ON d.player_id = p.player_id
    WHERE p.avg_points IS NOT NULL
    ORDER BY vorp DESC
    """
    
    return pd.read_sql(query, engine)


In [44]:
heatmap_df = calculate_all_vorp(engine)
heatmap_df.head(10)
heatmap_df.to_csv("../data/raw/heatmap_df.csv", index=False)

In [57]:
pivot_vorp = heatmap_df.pivot(index='roundPickNumber', columns='roundId', values='vorp')
pivot_player = heatmap_df.pivot(index='roundPickNumber', columns='roundId', values='player_name')
pivot_position = heatmap_df.pivot(index='roundPickNumber', columns='roundId', values='position')
pivot_points = heatmap_df.pivot(index='roundPickNumber', columns='roundId', values='points')

# Build hover text
hover_text = (
    '<b>' + pivot_player.fillna('') + '</b><br>' +
    'Pos: ' + pivot_position.fillna('') + '<br>' +
    'VORP: ' + pivot_vorp.round(2).astype(str) + '<br>' +
    'Points: ' + pivot_points.fillna(0).astype(int).astype(str)
)

# Create Heatmap
fig = go.Figure(data=go.Heatmap(
    z=pivot_vorp.values,
    x=pivot_vorp.columns,
    y=pivot_vorp.index,
    text=hover_text.values,
    hoverinfo="text",
    colorscale='RdYlGn',
    zmin=-10,
    zmax=15,
    showscale=True,
    colorbar=dict(title="VORP"),
))

# Add text in center of each box (VORP)
for i in range(len(pivot_vorp.index)):
    for j in range(len(pivot_vorp.columns)):
        value = pivot_vorp.iloc[i, j]
        if pd.notnull(value):
            fig.add_annotation(
                text=f"{value:.1f}",
                x=pivot_vorp.columns[j],
                y=pivot_vorp.index[i],
                showarrow=False,
                font=dict(color="black", size=10),
                xanchor="center",
                yanchor="middle"
            )

# Layout adjustments
fig.update_layout(
    title="VORP by Draft Pick (Interactive)",
    xaxis=dict(title="Draft Round", side='top', tickmode='linear', dtick=1),
    yaxis=dict(title="Pick Number in Round", autorange='reversed', tickmode='linear', dtick=1),
    width=1000,
    height=700,
    plot_bgcolor='white'
)

fig.write_html("../docs/charts/draft_value_chart.html")
fig.show()


<Figure size 640x480 with 0 Axes>

In [6]:

query = """
    SELECT  p.player_name,
            d.player_id,
            p.position, 
            d.lineupSlotId
    FROM draft d
    JOIN players p ON d.player_id = p.player_id
    WHERE p.position NOT IN ('QB', 'RB', 'WR', 'TE')
    LIMIT 50
"""


pd.read_sql(query, engine)

,player_name,player_id,position,lineupSlotId
0,CeeDee Lamb,4241389,BE,4
1,Jonathan Taylor,4242335,BE,2
2,Anthony Richardson,4429084,BE,0
3,Michael Pittman Jr.,4035687,BE,4
4,Chris Olave,4361370,BE,4
5,Drake London,4426502,BE,4
6,Alvin Kamara,3054850,BE,2
7,Dallas Goedert,3121023,BE,23
8,Nico Collins,4258173,BE,4
9,Rachaad White,4697815,BE,2


In [7]:
pivot = heatmap_df.pivot_table(
    values='VORP',
    index=''
)

KeyError: 'VORP'

In [ ]:
query = """
    WITH player_ppg AS (
    SELECT 
        player_id,
        position,
        points,
        games_played,
        (points * 1.0 / games_played) AS ppg
    FROM players
    WHERE games_played > 0 AND points IS NOT NULL
),

ranked_pos AS (
    SELECT *,
        RANK() OVER (PARTITION BY position ORDER BY ppg DESC) AS pos_rank
    FROM player_ppg
),

replacement_level AS (
    SELECT 
        position,
        AVG(ppg) AS replacement_ppg
    FROM ranked_pos
    WHERE pos_rank BETWEEN 15 AND 20 -- customize this range
    GROUP BY position
)

SELECT 
    r.player_id,
    r.position,
    r.ppg,
    p.player_name,
    rep.replacement_ppg,
    ROUND(r.ppg - rep.replacement_ppg, 2) AS vorp
FROM ranked_pos r
JOIN replacement_level rep ON r.position = rep.position
WHERE r.ppg IS NOT NULL
ORDER BY vorp DESC
    """
df = pd.read_sql(query, engine)
df


OperationalError: (sqlite3.OperationalError) no such column: p.player_name
[SQL: 
    WITH player_ppg AS (
    SELECT 
        player_id,
        position,
        points,
        games_played,
        (points * 1.0 / games_played) AS ppg
    FROM players
    WHERE games_played > 0 AND points IS NOT NULL
),

ranked_pos AS (
    SELECT *,
        RANK() OVER (PARTITION BY position ORDER BY ppg DESC) AS pos_rank
    FROM player_ppg
),

replacement_level AS (
    SELECT 
        position,
        AVG(ppg) AS replacement_ppg
    FROM ranked_pos
    WHERE pos_rank BETWEEN 15 AND 20 -- customize this range
    GROUP BY position
)

SELECT 
    r.player_id,
    r.position,
    r.ppg,
    p.player_name,
    rep.replacement_ppg,
    ROUND(r.ppg - rep.replacement_ppg, 2) AS vorp
FROM ranked_pos r
JOIN replacement_level rep ON r.position = rep.position
WHERE r.ppg IS NOT NULL
ORDER BY vorp DESC
    ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Focus on who was the best value in the fantasy league

In [ ]:
query = """
    WITH ranked_draft AS (
        SELECT
            player_id,
            RANK() OVER (ORDER BY avg ASC) AS overallPickNumber
        FROM draft
    ),
    ranked_points AS (
        SELECT
            player_id,
            points,
            RANK() OVER (ORDER BY points DESC) AS performance_rank
        FROM players
        WHERE points IS NOT NULL
    )

    SELECT
        p.player_name,
        p.position,
        d.overallPickNumber,
        pr.performance_rank,
        pr.points,
        p.current_team_name,
        (d.overallPickNumber - pr.performance_rank) AS value_score
    FROM draft d
    JOIN ranked_points pr ON d.player_id = pr.player_id
    JOIN players p ON p.player_id = d.player_id
    ORDER BY value_score DESC
    """
df = pd.read_sql(query, engine)
df



,player_name,position,overallPickNumber,performance_rank,points,current_team_name,value_score
0,Bo Nix,BE,214,15,300,Mendes Army,199
1,Geno Smith,QB,207,32,250,Jean Machine,175
2,Baker Mayfield,QB,169,5,352,South Bay Starr Power,164
3,Chuba Hubbard,RB,202,46,232,Ambler Thighs,156
4,Brian Thomas Jr.,RB/WR/TE,154,20,275,The Ralph Dudes,134
...,...,...,...,...,...,...,...
219,Isiah Pacheco,RB,8,314,52,Deb’s Debsters,-306
220,MarShawn Lloyd,None,167,481,2,FA,-314
221,Zamir White,BE,62,391,26,Ambler Thighs,-329
222,Christian McCaffrey,None,1,337,43,FA,-336


Who had the most value per ppg

In [ ]:
pd.read_sql("SELECT * FROM players LIMIT 5", engine)
pd.read_sql("SELECT * FROM average_draft_position LIMIT 5", engine)
pd.read_sql("SELECT * FROM draft LIMIT 5", engine)


,player_id,overallPickNumber,team_id,roundPickNumber,id,roundId,autoDraftTypeId,lineupSlotId
0,3117251,1,3,1,1,1,0,2
1,4241389,2,14,2,2,1,0,4
2,3929630,3,1,3,3,1,0,2
3,3918298,4,4,4,4,1,0,0
4,4427366,5,8,5,5,1,3,2
